In [ ]:
# from pyspark.sql import functions as F, Window, types as T
# from delta.tables import DeltaTable
# from datetime import datetime

# # 1. Lire chaque JSON brut en un seul bloc
# raw = (spark.read.option("wholetext", True).text("Files/bronze/weather/*.json")
#        .withColumn("filepath", F.input_file_name()))

# # 2. Ville et date d'ingestion, lues dans le nom du fichier (Ville_yyyyMMddHHmm.json)
# raw = (raw
#     .withColumn("city", F.regexp_extract("filepath", r"([A-Za-z]+)_\d{12}\.json", 1))
#     .withColumn("ingestion_ts", F.to_timestamp(
#         F.regexp_extract("filepath", r"_(\d{12})\.json", 1), "yyyyMMddHHmm")))

# # 3. Schéma du JSON Open-Meteo (4 variables)
# arr = lambda t: T.ArrayType(t)
# schema = T.StructType([T.StructField("hourly", T.StructType([
#     T.StructField("time", arr(T.StringType())),
#     T.StructField("temperature_2m", arr(T.DoubleType())),
#     T.StructField("relative_humidity_2m", arr(T.DoubleType())),
#     T.StructField("precipitation", arr(T.DoubleType())),
#     T.StructField("wind_speed_10m", arr(T.DoubleType())),
# ]))])

# # 4. Un seul explode : arrays_zip garde l'alignement heure / valeurs
# parsed = (raw.select("city", "ingestion_ts", F.from_json("value", schema).alias("d"))
#     .select("city", "ingestion_ts", F.explode(F.arrays_zip(
#         "d.hourly.time", "d.hourly.temperature_2m", "d.hourly.relative_humidity_2m",
#         "d.hourly.precipitation", "d.hourly.wind_speed_10m")).alias("z"))
#     .select("city", "ingestion_ts",
#         F.to_timestamp("z.time").alias("time"),
#         F.col("z.temperature_2m").alias("temperature_2m"),
#         F.col("z.relative_humidity_2m").alias("relative_humidity_2m"),
#         F.col("z.precipitation").alias("precipitation"),
#         F.col("z.wind_speed_10m").alias("wind_speed_10m")))

# # 5. Dédoublonnage : on garde la prévision du fichier le plus récent
# w = Window.partitionBy("city", "time").orderBy(F.col("ingestion_ts").desc())
# latest = parsed.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

# # 6. Contrôle qualité : valeurs plausibles, sinon rejetées
# is_valid = F.coalesce(
#     F.col("time").isNotNull()
#     & F.col("temperature_2m").between(-50, 60)
#     & F.col("relative_humidity_2m").between(0, 100)
#     & (F.col("precipitation") >= 0)
#     & (F.col("wind_speed_10m") >= 0), F.lit(False))
# df_valid = latest.filter(is_valid)

# # 7. Journal de qualité (une ligne par exécution)
# n_read, n_dedup, n_valid = parsed.count(), latest.count(), df_valid.count()
# spark.createDataFrame(
#     [(datetime.utcnow(), n_read, n_read - n_dedup, n_dedup - n_valid, n_valid)],
#     "run_ts timestamp, rows_read long, duplicates_removed long, rows_rejected long, rows_written long"
# ).write.format("delta").mode("append").saveAsTable("silver_quality_log")

# # 8. MERGE (upsert) vers la table Silver
# if spark.catalog.tableExists("silver_weather"):
#     (DeltaTable.forName(spark, "silver_weather").alias("t")
#         .merge(df_valid.alias("s"), "t.city = s.city AND t.time = s.time")
#         .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
# else:
#     df_valid.write.format("delta").saveAsTable("silver_weather")

# print(f"Lues: {n_read} | doublons: {n_read - n_dedup} | rejetées: {n_dedup - n_valid} | écrites: {n_valid}")

In [ ]:
from pyspark.sql import functions as F, Window, types as T
from delta.tables import DeltaTable
from datetime import datetime, timezone
import re

BRONZE = "Files/bronze/weather"
LOG_SCHEMA = ("run_ts timestamp, files_read long, rows_read long, duplicates_removed long, "
              "rows_rejected long, rows_written long, missing_cities string, max_file_ts timestamp")

def log_run(files, read, dups, rejected, written, missing, max_ts):
    (spark.createDataFrame(
        [(datetime.now(timezone.utc), files, read, dups, rejected, written, ",".join(missing), max_ts)],
        LOG_SCHEMA)
     .write.format("delta").mode("append").option("mergeSchema", "true")
     .saveAsTable("silver_quality_log"))

# 0. Point de reprise : dernier fichier traité AVEC SUCCÈS (lu dans le journal)
last_ts = None
if spark.catalog.tableExists("silver_quality_log") and \
        "max_file_ts" in spark.table("silver_quality_log").columns:
    last_ts = spark.table("silver_quality_log").agg(F.max("max_file_ts")).collect()[0][0]
if last_ts is None and spark.catalog.tableExists("silver_weather"):   # 1er run après la migration
    last_ts = spark.table("silver_weather").agg(F.max("ingestion_ts")).collect()[0][0]

# 1. Lister le Bronze, ne garder que les nouveaux fichiers
def file_ts(name):
    m = re.search(r"_(\d{12})\.json$", name)
    return datetime.strptime(m.group(1), "%Y%m%d%H%M") if m else None

all_files = [f for f in notebookutils.fs.ls(BRONZE) if f.name.endswith(".json")]
new_files = [f for f in all_files if file_ts(f.name) and (last_ts is None or file_ts(f.name) > last_ts)]
print(f"Fichiers Bronze : {len(all_files)} | nouveaux : {len(new_files)}")

if not new_files:
    log_run(0, 0, 0, 0, 0, [], None)
    notebookutils.notebook.exit("Aucun nouveau fichier")

# 2. Lecture des nouveaux fichiers
raw = (spark.read.option("wholetext", True).text([f.path for f in new_files])
       .withColumn("filepath", F.input_file_name())
       .withColumn("city", F.regexp_extract("filepath", r"([A-Za-z]+)_\d{12}\.json", 1))
       .withColumn("ingestion_ts", F.to_timestamp(
           F.regexp_extract("filepath", r"_(\d{12})\.json", 1), "yyyyMMddHHmm")))

arr = lambda t: T.ArrayType(t)
schema = T.StructType([T.StructField("hourly", T.StructType([
    T.StructField("time", arr(T.StringType())),
    T.StructField("temperature_2m", arr(T.DoubleType())),
    T.StructField("relative_humidity_2m", arr(T.DoubleType())),
    T.StructField("precipitation", arr(T.DoubleType())),
    T.StructField("wind_speed_10m", arr(T.DoubleType())),
]))])

parsed = (raw.select("city", "ingestion_ts", F.from_json("value", schema).alias("d"))
    .select("city", "ingestion_ts", F.explode(F.arrays_zip(
        "d.hourly.time", "d.hourly.temperature_2m", "d.hourly.relative_humidity_2m",
        "d.hourly.precipitation", "d.hourly.wind_speed_10m")).alias("z"))
    .select("city", "ingestion_ts",
        F.to_timestamp("z.time").alias("time"),
        F.col("z.temperature_2m").alias("temperature_2m"),
        F.col("z.relative_humidity_2m").alias("relative_humidity_2m"),
        F.col("z.precipitation").alias("precipitation"),
        F.col("z.wind_speed_10m").alias("wind_speed_10m")))

# 3. Complétude : toutes les villes de cities.csv sont-elles arrivées ?
expected = {r["city"] for r in spark.read.option("header", True).csv("Files/config/cities.csv").collect()}
received = {r["city"] for r in parsed.select("city").distinct().collect()}
missing = sorted(expected - received)

# 4. Dédoublonnage (prévision la plus récente) + contrôle des valeurs
w = Window.partitionBy("city", "time").orderBy(F.col("ingestion_ts").desc())
latest = parsed.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
is_valid = F.coalesce(
    F.col("time").isNotNull()
    & F.col("temperature_2m").between(-50, 60)
    & F.col("relative_humidity_2m").between(0, 100)
    & (F.col("precipitation") >= 0) & (F.col("wind_speed_10m") >= 0), F.lit(False))
df_valid = latest.filter(is_valid)
n_read, n_dedup, n_valid = parsed.count(), latest.count(), df_valid.count()
dups, rejected = n_read - n_dedup, n_dedup - n_valid

# 5. Ville manquante : journal d'échec (sans avancer le point de reprise), puis échec
if missing:
    log_run(len(new_files), n_read, dups, rejected, 0, missing, None)
    raise Exception(f"Villes manquantes dans ce run : {missing}")

# 6. MERGE conditionnel
cols = ["temperature_2m", "relative_humidity_2m", "precipitation", "wind_speed_10m"]
changed = " OR ".join(f"NOT (t.{c} <=> s.{c})" for c in cols)
if spark.catalog.tableExists("silver_weather"):
    (DeltaTable.forName(spark, "silver_weather").alias("t")
        .merge(df_valid.alias("s"), "t.city = s.city AND t.time = s.time")
        .whenMatchedUpdateAll(condition=changed)
        .whenNotMatchedInsertAll().execute())
else:
    df_valid.write.format("delta").saveAsTable("silver_weather")

# 7. Journal de succès : c'est ICI que le point de reprise avance
log_run(len(new_files), n_read, dups, rejected, n_valid, [], max(file_ts(f.name) for f in new_files))
print(f"Nouveaux fichiers : {len(new_files)} | lues : {n_read} | doublons : {dups} | traitées : {n_valid}")

****GOLD****

In [ ]:
from pyspark.sql import functions as F

gold = (spark.table("silver_weather").withColumn("date", F.to_date("time"))
    .groupBy("city", "date").agg(
        F.round(F.avg("temperature_2m"), 1).alias("temp_moyenne"),
        F.round(F.max("temperature_2m"), 1).alias("temp_max"),
        F.round(F.min("temperature_2m"), 1).alias("temp_min"),
        F.round(F.avg("relative_humidity_2m"), 1).alias("humidite_moyenne"),
        F.round(F.sum("precipitation"), 1).alias("precipitation_totale"),
        F.round(F.max("wind_speed_10m"), 1).alias("vent_max")))


(gold.write.format("delta").mode("overwrite")
     .option("overwriteSchema", "true").saveAsTable("gold_weather_daily"))


In [ ]:
spark.table("gold_weather_daily").printSchema()
spark.sql("SELECT city, COUNT(*) jours FROM gold_weather_daily GROUP BY city").show()